# The Plan and Execute Agent

The ReAct agent is a good starting point for building agents. It has limited planning capabilities, as it can only plan one step ahead. For more complex tasks, we need an agent that can create a multi-step plan and then execute it step by step. In this notebook, we will implement a Plan and Execute agent that can do just that. The Plan and Execute agent is capable of creating a plan and can do replanning if necessary.

In [ ]:
import os
import dotenv
import json
from IPython.display import Image
from typing import List, Annotated, Tuple, Union
from operator import add
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool, Tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent
from langfuse.langchain import CallbackHandler

# Load environment variables from .env file.
dotenv.load_dotenv()

# Initialize the Langfuse handler
langfuse_handler = CallbackHandler()

# Initialize the chat model
chat_model_string = os.environ["LANGCHAIN_CHAT_MODEL_ANTHROPIC"]

As always, we have imported the necessary libraries and set up the environment.

We will recycle the same tools as before. Keep in mind that you can always add more and different tools to your agent.

In [ ]:
class MathToolInput(BaseModel):
    """Input for the math tool."""
    operand1: float = Field(..., description="The first operand for the math operation.")
    operand2: float = Field(..., description="The second operand for the math operation.")
    operator: str = Field(..., description="The operator for the math operation", enumerate=["+", "-", "*", "/"])

@tool
def math_tool(input: MathToolInput) -> float:
    """Perform a math operation based on the input."""
    print(f"Performing math operation: {input.operand1} {input.operator} {input.operand2}")
    if input.operator == "+":
        return input.operand1 + input.operand2
    elif input.operator == "-":
        return input.operand1 - input.operand2
    elif input.operator == "*":
        return input.operand1 * input.operand2
    elif input.operator == "/":
        if input.operand2 == 0:
            raise ValueError("Cannot divide by zero.")
        return input.operand1 / input.operand2
    else:
        raise ValueError(f"Unknown operator: {input.operator}")


class SortToolInput(BaseModel):
    """Input for the sort tool."""
    numbers: List[float] = Field(..., description="A list of numbers to sort.")
    order: str = Field(..., description="The order to sort the numbers", enumerate=["asc", "desc"])

@tool
def sort_tool(input: SortToolInput) -> List[float]:
    """Sort a list of numbers in ascending or descending order."""
    print(f"Sorting numbers: {input.numbers} in {input.order} order")
    if input.order == "asc":
        return sorted(input.numbers)
    elif input.order == "desc":
        return sorted(input.numbers, reverse=True)
    else:
        raise ValueError(f"Unknown order: {input.order}")
    
tools: List[Tool] = [
    math_tool,
    sort_tool
]

As always, the agent has its own state. While the ReAct agent only needed to keep track of the past messages, the Plan and Execute agent needs to keep track of a few other things.

In [ ]:
class PlanAndExecuteState(BaseModel):
    """State for the plan and execute agent."""

    input: str = Field(
        ...,
        description="The input for the agent."
    )
    plan: List[str] = Field(
        default_factory=list,
        description="The plan generated by the agent."
    )
    past_steps: Annotated[List[Tuple], add] = Field(
        default_factory=list,
        description="The past steps taken by the agent."
    )
    response: str = Field(
        default="",
        description="The response generated by the agent."
    )

The `input` is the user's input. This is a string that triggers the agent to start planning and executing.

The `plan` field is the current plan of the agent. This is a list of steps that the agent needs to execute in order to complete the task. 

The `past_steps` field is a list of steps that the agent has already executed. This is useful for the agent to keep track of what it has done so far and what was successful or not.

The `response` field is the final response that the agent will give to the user. Initially empty, it will be filled once the agent has completed its task.

It is time for a little trick. It turns out that the ReAct agent can be used to implement the planning step execution part of the Plan and Execute agent. Thus we will instantiate a ReAct agent and give it access to the tools.

The cool thing is that LangGraph works recursively. This means that we can use an agent, agentic workflows or workflows as subcomponents of other agents, agentic workflows or workflows. This is a powerful feature that allows us to build complex systems out of simple components.

In [ ]:
# Create an agent to execute the next step of the plan.
model = init_chat_model(chat_model_string)
prompt = "You are a helpful assistant."
execute_agent = create_react_agent(
    model,
    tools,
    prompt=prompt
)

## Planning

The initial step is to create a plan based on the user's input. This plan will consist of multiple steps that the agent will execute one by one.

We will define a Pydantic model to represent the plan. The plan will contain a list of steps, each step being a string that describes what the agent needs to do.

We will also add a planning node to our workflow. This node will take the current state of the agent as input and will output the updated state with the new plan.

In [ ]:
class Plan(BaseModel):
    """A plan generated by the agent."""
    steps: List[str] = Field(
        default_factory=list,
        description="The steps in the plan. Sorted by their execution order. Should contain at least one step and as many steps as necessary."
    )

def plan_node(state: PlanAndExecuteState) -> PlanAndExecuteState:

    # Create a model with structured output.
    model_with_structured_output = init_chat_model(chat_model_string).with_structured_output(Plan)

    # Create the system message.
    system_message_content = "You are a helpful assistant specialized on planning."
    system_message_content += " Given a goal, create a straightforward sequential plan broken down into individual tasks."
    system_message_content += " When completed properly, these tasks should lead to the correct solution."
    system_message_content += " Keep the plan focused—avoid unnecessary steps."
    system_message_content += " Each step should contain all required information without gaps, and the last step should produce the final answer."
    for tool in tools:
        system_message_content += f"\nTool available: {tool.name}. Description: `{tool.description}`."
    system_message = SystemMessage(
        content=system_message_content
    )
    
    # Create the human message.
    human_message = HumanMessage(
        content=state.input
    )

    # Invoke the model.
    plan = model_with_structured_output.invoke(
        [system_message, human_message]
    )
    print(f"Generated plan: {json.dumps(plan.model_dump(), indent=2)}")

    # Return the update.
    return {
        "plan": plan.steps
    }

In a nutshell, the planning node is an LLM call that will generate a plan based on the user's input. The prompt for the LLM will include the user's input and a description of the tools available to the agent. Pay special attention to our use of structured output here. This is crucial to ensure that the LLM outputs a valid plan that can be parsed and used by the agent. Once the LLM call is made, the state will be updated with the new plan.

# Execution

The execution part of the Plan and Execute agent is responsible for carrying out the steps in the plan. This involves executing each step one by one, using the tools available to the agent. We will introduce an execution node in our workflow that will handle this process.

In [ ]:
def execute_node(state: PlanAndExecuteState) -> PlanAndExecuteState:
    
    # Unpack the necessary information from the state.
    plan = state.plan
    first_step = plan[0]

    # Create the human message.
    human_message_content = "Here is a plan:"
    for step_index, step in enumerate(plan):
        human_message_content += f"\n{step_index + 1}. {step}"
    human_message_content += f"\n\nNow execute the first step of the plan: `{first_step}`."
    human_message = HumanMessage(
        content=human_message_content
    )

    # Invoke the agent.
    agent_response = execute_agent.invoke({
        "messages": [human_message]
    })
    agent_response_content = agent_response["messages"][-1].content

    # Return the update.
    past_step_update = (first_step, agent_response_content)
    return {
        "past_steps": [past_step_update]
    }

You see that we are using the ReAct agent we defined earlier to execute the steps in the plan. The execution node will take the current state of the agent as input, including the plan and the past steps, and will output an updated state with the executed steps. This is the memory of the past.

## Replanning

A strength of the Plan and Execute agent is its ability to replan if necessary. If a step in the plan fails or if the agent encounters an unexpected situation, it can go back to the planning phase and create a new plan based on the current state. Of course, replanning could lead to the conclusion that it does not make sense to continue. This might be the case if we have reached a dead end of, of cource, that the task has been completed.

In [ ]:
class ResponseToUser(BaseModel):
    """Response to the user."""
    
    response: str = Field(
        ...,
        description="The response to the user."
    )

class Action(BaseModel):
    """Action to take."""
    
    action: Union[ResponseToUser, Plan] = Field(
        description="An action to perform. Either respond to the user using `ResponseToUser` or if you want to continue by using more tools create a `Plan`."
    )

def replan_node(state: PlanAndExecuteState) -> PlanAndExecuteState:

    # Create the model.
    model_with_structured_output = init_chat_model(chat_model_string).with_structured_output(Action)

    # Create the human message.
    human_message_content = "Create a straightforward sequential approach for this objective, breaking it into specific tasks that will lead to the correct solution when executed properly. Keep it streamlined—include only necessary steps. Each step should contain complete information without gaps, and the final step should produce the end result."
    human_message_content += f"\n\nYour goal is: `{state.input}`."
    human_message_content += f"\n\nYour initial plan was: `{state.plan}`."
    human_message_content += f"\n\nSo far, you have completed these actions: `{state.past_steps}`."
    human_message_content += f"\n\nRevise your approach based on current progress. If all necessary work is complete and you're ready to provide the final response, indicate that. Otherwise, outline the remaining tasks. Only include steps that still require completion—don't repeat already finished work."
    human_message = HumanMessage(
        content=human_message_content
    )

    # Invoke the model.
    action = model_with_structured_output.invoke(
        [human_message]
    )
    print(f"Revised plan: {json.dumps(action.model_dump(), indent=2)}")

    # Return with a response to the user.
    if isinstance(action.action, ResponseToUser):
        return {
            "response": action.action.response
        }
    
    # Return with an updated plan.
    elif isinstance(action.action, Plan):
        return {
            "plan": action.action.steps
        }

We reuse the `Plan` data model we defined earlier to represent the plan. On top of that, we introduce a `ResponseToUser` data model to represent the final response that the agent will give to the user. The `Action` class is a little tricky: It enabled the LLM to either generate a new plan or to generate a final response to the user. Thus the decision whether to replan or to finish the task is left to the LLM.

The `replan_node` itself is very similar to the `planning_node`. The main difference is that it takes the current state of the agent as input, including the past steps and the current plan. The prompt for the LLM will include this information, allowing the LLM to make an informed decision about whether to replan or to finish the task.

## Deciding when to stop.

The decision when to stop is fairly easy. If the LLM has generated a final response to the user, we stop. If not, we continue.

In [ ]:
def should_end(state: PlanAndExecuteState) -> bool:
    if state.response and state.response != "":
        return END
    else:
        return "execute_node"

## Putting it all together

We have all the nodes in place. Now we can put everything together in a workflow. The workflow will start with the user's input, then it will create a plan, execute the plan, and replan if necessary. The workflow will continue until the agent has completed its task and has a final response for the user.

In [ ]:
# Create the graph using the state.
graph = StateGraph(PlanAndExecuteState)

# Add the nodes.
graph.add_node(plan_node)
graph.add_node(execute_node)
graph.add_node(replan_node)

# Add the edges.
graph.add_edge(START, "plan_node")
graph.add_edge("plan_node", "execute_node")
graph.add_edge("execute_node", "replan_node")

# Add a conditional edge from the replan node back to the plan node or the end state.
graph.add_conditional_edges(
    "replan_node",
    should_end,
    ["execute_node", END]
)

# Create the agent.
plan_and_execute_agent = graph.compile()
display(Image(plan_and_execute_agent.get_graph(xray=True).draw_mermaid_png()))

We see two loops in here. As we have already learned, the ReAct agent has a loop to execute the steps in the plan. On top of that, we have another loop that allows the agent to replan if necessary. The outer loop will continue until the agent has a final response for the user.

Let us finish by testing our Plan and Execute agent. We will give it a complex task that requires multiple steps to complete.

In [ ]:
result = plan_and_execute_agent.invoke(
    {
        "input": "Generate 4 examples that use the math tool and the sort tool. After that run the tools."
    },
    config={"callbacks": [langfuse_handler]}
)
print(result)

# Done.